In [ ]:
import os, mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from itertools import product

from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import f1_score
from decodingFuncs import filter_and_downsample, generate_pc_timeseries, runMCC_singleChannel

from matplotlib.gridspec import GridSpec

%matplotlib qt
%load_ext autoreload
%autoreload 2

In [ ]:
bidsRoot = '/System/Volumes/Data/d/DATD/datd/MEG_MGS/MEG_BIDS'
taskName = 'mgs'

subjID = 1
subName = 'sub-%02d' % subjID
derivativesRoot = os.path.join(bidsRoot, 'derivatives', subName, 'meg')
stimRoot = os.path.join(bidsRoot, subName, 'stimfiles')
fNameRoot = subName + '_task-' + taskName
stimLocked_fpath = os.path.join(derivativesRoot, fNameRoot + '_stimlocked.mat')
TFR_fpath = os.path.join(derivativesRoot, fNameRoot + '_TFR.mat')

# Load stimlocked data and TFR data
epocStimLocked = loadmat(stimLocked_fpath)
TFRmat = loadmat(TFR_fpath)

In [ ]:
# Good chans
leftChans = ['AG009', 'AG010', 'AG011', 'AG012', 'AG013', 'AG014', 'AG015', 'AG016', 'AG025', 'AG026', 'AG027', 'AG028', 'AG029', 'AG030', 'AG032', 'AG041', 'AG042', 'AG043', 'AG045', 'AG048', 'AG057', 'AG059', 'AG060', 'AG062', 'AG063', 'AG073', 'AG075', 'AG078', 'AG079', 'AG091', 'AG092', 'AG106', 'AG110', 'AG111']
rightChans = ['AG001', 'AG002', 'AG003', 'AG004', 'AG006', 'AG007', 'AG008', 'AG018', 'AG019', 'AG020', 'AG021', 'AG022', 'AG024', 'AG034', 'AG036', 'AG037', 'AG038', 'AG039', 'AG040', 'AG049', 'AG050', 'AG051', 'AG052', 'AG053', 'AG054', 'AG055', 'AG056', 'AG058', 'AG065', 'AG081', 'AG082', 'AG085', 'AG097', 'AG098', 'AG103', 'AG104', 'AG113']

In [ ]:
# Create an outline of the gradiometers
ch_label_array = TFRmat['TFRleft_power'][0][0][0]
ch_label = []
for i in range(len(ch_label_array)):
    ch_label.append(ch_label_array[i][0][0])
dimord = TFRmat['TFRleft_power'][0][0][1]
freqs = TFRmat['TFRleft_power'][0][0][2][0]
times = TFRmat['TFRleft_power'][0][0][3][0]
# grad = TFRmat['TFRleft_power'][0][0][4][0]
trialInfo_left = TFRmat['TFRleft_power'][0][0][5]
powspctrm_left = TFRmat['TFRleft_power'][0][0][7]
trialInfo_right = TFRmat['TFRright_power'][0][0][5]
powspctrm_right = TFRmat['TFRright_power'][0][0][7]

In [ ]:
classCats='quadrant'
freqband='beta'
v73 = False
accuracy_matrix, f1score_matrix, times_crop = runMCC_singleChannel(TFRmat, v73, classCats, freqband)

In [ ]:
plt.figure()
plt.hist(accuracy_matrix.flatten(), bins=100)
plt.show()

In [ ]:
plt.figure()
# plt.plot(times_crop, np.mean(accuracy_matrix, axis=0), linewidth=2)
# plt.plot(times_crop, accuracy_matrix[:, :, 0].T, 'k', alpha=0.1)
plt.plot(times_crop, np.mean(f1score_matrix, axis=0), linewidth=2)
plt.plot(times_crop, f1score_matrix[:, :, 0].T, 'k', alpha=0.1)
plt.show()

In [ ]:
from matplotlib.widgets import Slider
def plotTopo(scoreMat, grad, times_crop, init_time=0.5, plotType='3D'):
    thTimeIdx = np.argmin(np.abs(times_crop - init_time))
    thScoreFirst = scoreMat[:, thTimeIdx, 0]
    colorThresh = 0.17
    colorVecFirst = ['r' if f1 > colorThresh else 'w' for f1 in thScoreFirst]
    if plotType == '3D':
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')
        sc = ax.scatter(grad[:, 0], grad[:, 1], grad[:, 2], c=colorVecFirst, s=300, marker='o')
        plt.subplots_adjust(bottom=0.25)
        ax_slider = plt.axes([0.25, 0.1, 0.65, 0.03])
        slider = Slider(
            ax=ax_slider,
            label='Time',
            valmin=times_crop[0],
            valmax=times_crop[-1],
            valinit=init_time,
            valstep=np.diff(times_crop).min()
        )
    elif plotType == '2D':
       raise NotImplementedError
    
    def update(val):
        t = slider.val
        time_idx = np.argmin(np.abs(times_crop - t))
        thScore = scoreMat[:, time_idx, 0]
        newColorVec = ['r' if f1 > colorThresh else 'w' for f1 in thScore]
        sc.set_color(newColorVec)
        fig.canvas.draw_idle()
    
    slider.on_changed(update)
    plt.show()
    

grad = TFRmat['TFRleft_power'][0][0][4][0][0][3]
grad = np.array(grad)

plotTopo(f1score_matrix, grad, times_crop, init_time=0.1, plotType='3D')

In [ ]:
# Extract features from the data that are relevant for plotting
# For stimlocked data, there is one key stimLocked and it has the following structure:
#      0: ch_label
#      1: trialinfo
#      2: sampleinfo
#      3: grad structure
#      4: trial
#      5: time
#      6: fsample
#      7: cfg
# ch_label = epocStimLocked['epocStimLocked'][0][0][0]
# trialinfo = epocStimLocked['epocStimLocked'][0][0][1]
# sampleinfo = epocStimLocked['epocStimLocked'][0][0][2]
# trial = epocStimLocked['epocStimLocked'][0][0][4][0]
# trials_list = [np.array(t) for t in trial]
# trial_arr = np.stack(trials_list, axis=0)
# time = epocStimLocked['epocStimLocked'][0][0][5]
# time_arr = time[0][0][0]
# Fs = epocStimLocked['epocStimLocked'][0][0][6][0][0]



# for TFR data, there are 2 keys TFRleft_power and TFRright_power and each of them have the following structure:
#      0: ch_label
#      1: dimord
#      2: freqs
#      3: times
#      4: grad structure
#      5: trialinfo
#      6: cfg
#      7: powspctrm
ch_label_array = TFRmat['TFRleft_power'][0][0][0]
ch_label = []
for i in range(len(ch_label_array)):
    ch_label.append(ch_label_array[i][0][0])
dimord = TFRmat['TFRleft_power'][0][0][1]
freqs = TFRmat['TFRleft_power'][0][0][2][0]
times = TFRmat['TFRleft_power'][0][0][3][0]
trialInfo_left = TFRmat['TFRleft_power'][0][0][5]
powspctrm_left = TFRmat['TFRleft_power'][0][0][7]
trialInfo_right = TFRmat['TFRright_power'][0][0][5]
powspctrm_right = TFRmat['TFRright_power'][0][0][7]

# Outlining though process:
#### 1. Combine the power spectra and trial info
#### 2. Average data across freq bands: theta, alpha, beta for each trial, channel, time-point
#### 3. reshape data to: (trials, [chans_theta, chans_alpha, chans_beta], timeponts) # Maybe even add a broadband estimate, figure out how best to do that?
#### 4. Average across trials by conditions, let's start with hemifield so left vs right, data will be: (nconds, [chans_theta, chans_alpha, chans_beta], timeponts)
#### 5. Choose timepoint = 0, and you have: (nconds, [chans_theta, chans_alpha, chans_beta])
#### 6. Perform PCA over channels & frequencies at time = 0, check variance explained and the loadings of the top 2 PCs
#### 7. Apply the PCA (assuming it captures most variance, to all other time-points)

In [ ]:
powspctrm_left.shape

In [ ]:
# Find Idx for left and right channels
leftIdx = [ch_label.index(ch) for ch in leftChans]
rightIdx = [ch_label.index(ch) for ch in rightChans]
chIdx = leftIdx + rightIdx
print(len(chIdx))

In [ ]:
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

# Combine the data from left and right groups
trialInfo_combined = np.concatenate((trialInfo_left, trialInfo_right), axis=0)
powspctrm_combined = np.concatenate((powspctrm_left, powspctrm_right), axis=0)

# Select only the relevant channels
powspctrm_combined = powspctrm_combined[:, leftIdx, :, :]

# baselineTime = (-1, 0)
# baselineIdx = np.where((times >= baselineTime[0]) & (times < baselineTime[1]))[0]
# powspctrm_unlogged = 10**(powspctrm_combined / 10)
# baseline = np.nanmean(powspctrm_unlogged[:, :, :, baselineIdx], axis=-1)
# baseline = np.repeat(baseline[:, :, :, np.newaxis], powspctrm_unlogged.shape[-1], axis=-1)
# powspctrm_combined = 10*np.log10(powspctrm_unlogged / baseline)
# powspctrm_combined = powspctrm_unlogged / baseline

time_relevIdx = np.where((times >= -1) & (times <= 2))[0]   
time_relev = times[time_relevIdx]
powspctrm_combined = powspctrm_combined[:, :, :, time_relevIdx]

theta_freqs = np.where((freqs >= 4) & (freqs <= 8))[0]
alpha_freqs = np.where((freqs >= 8) & (freqs <= 12))[0]
beta_freqs = np.where((freqs >= 12) & (freqs <= 35))[0]

theta_power = np.nanmean(powspctrm_combined[:, :, theta_freqs, :], axis=2)
alpha_power = np.nanmean(powspctrm_combined[:, :, alpha_freqs, :], axis=2)
beta_power = np.nanmean(powspctrm_combined[:, :, beta_freqs, :], axis=2)

multiband_power = np.concatenate((theta_power, alpha_power, beta_power), axis=1)  

n_groups = 4
n_comps = 3
tinfo = trialInfo_combined[:, 1]
data = alpha_power
if n_groups == 4:
    groups =  {
        'Q1': np.where((tinfo == 2) | (tinfo == 3))[0],
        'Q2': np.where((tinfo == 4) | (tinfo == 5))[0],
        'Q3': np.where((tinfo == 7) | (tinfo == 8))[0],
        'Q4': np.where((tinfo == 9) | (tinfo == 10))[0],
    }
elif n_groups == 6:
    groups = {
        'T1': np.where(tinfo == 1)[0],
        'T2': np.where((tinfo == 2) | (tinfo == 3))[0],
        'T3': np.where((tinfo == 4) | (tinfo == 5))[0],
        'T4': np.where((tinfo == 6))[0],
        'T5': np.where((tinfo == 7) | (tinfo == 8))[0],
        'T6': np.where((tinfo == 9) | (tinfo == 10))[0],
    }
elif n_groups == 10:
    groups = {
        'T1': np.where(tinfo == 1)[0],
        'T2': np.where(tinfo == 2)[0],
        'T3': np.where(tinfo == 3)[0],
        'T4': np.where(tinfo == 4)[0],
        'T5': np.where(tinfo == 5)[0],
        'T6': np.where(tinfo == 6)[0],
        'T7': np.where(tinfo == 7)[0],
        'T8': np.where(tinfo == 8)[0],
        'T9': np.where(tinfo == 9)[0],
        'T10': np.where(tinfo == 10)[0],
    }
elif n_groups == 2:
    groups =  {
        'right': np.where((tinfo == 2) | (tinfo == 3) | (tinfo == 1) | (tinfo == 9) | (tinfo == 10))[0],
        'left': np.where((tinfo == 4) | (tinfo == 5) | (tinfo == 6) | (tinfo == 7) | (tinfo == 8))[0],
    }

# Average data for each group
averaged_data = {}
for grp_name, indices in groups.items():
    averaged_data[grp_name] = np.nanmean(data[indices, :, :], axis=0) # (nchans, ntimes)


# Create matrix D at t=0
# tRelevIdx = np.where(np.isclose(time_relev, 0.2, atol=1e-2))[0][0]
tRelevIdx = np.where((time_relev >= 0) & (time_relev <= 0.5))[0]
# D = np.array([avg[:, tRelevIdx] for avg in averaged_data.values()]) # (ngroups, nchans)
D = np.array([np.mean(avg[:, tRelevIdx], axis=1) for avg in averaged_data.values()]) # (ngroups, nchans)
# # Demean columns
D = D - np.mean(D, axis=0)


# Perform PCA on D
pca = PCA(n_components=n_comps)
# pca = KernelPCA(n_components=n_comps, kernel='rbf')
pc_comps = pca.fit_transform(D) # (ngroups, ncomponents)
# Print explained variance
print(pca.explained_variance_ratio_)

# Apply PCA to every time point
pc_timeseries = np.zeros((len(groups), n_comps, data.shape[2]))
for t in range(data.shape[2]):
    D_t = np.array([avg[:, t] for avg in averaged_data.values()])
    # D_t = np.array([np.mean(avg[:, t], axis=1) for avg in averaged_data.values()])
    D_t = D_t - np.mean(D_t, axis=0)
    
    pc_timeseries[:, :, t] = pca.transform(D_t)

In [ ]:
plt.figure()
plt.imshow(pca.components_, aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
# Create a 3D plot of the first 2 PCs
fig = plt.figure(figsize = (12, 8))
ax = fig.add_subplot(111, projection='3d')
# color_list = ['r', 'g', 'b', 'y']
color_list = plt.cm.tab10.colors

colors = [color_list[i] for i in range(len(groups))]
time_pts = time_relev

for group_idx in range(len(groups)):
    x = pc_timeseries[group_idx, 0, :]
    y = pc_timeseries[group_idx, 1, :]
    z = time_pts

    sc = ax.scatter3D(
        x, y, z,
        c=colors[group_idx],
        marker='o',
        s=40,
        edgecolors=colors[group_idx],
        label=list(groups.keys())[group_idx]
    )

ax.plot([0, 0], [0, 0], [z.min(), z.max()], 'k--', alpha=0.5)
ax.plot([0, 0], [0, 0], [0, 0], 'ko', ms=8)

ax.set_xlabel('PC1', fontsize=12, labelpad=15)
ax.set_ylabel('PC2', fontsize=12, labelpad=15)
ax.set_zlabel('Time (s)', fontsize=12, labelpad=15)
ax.xaxis.pane.fill = True
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.grid(True, alpha=0.3)

ax.legend(loc='upper left', fontsize=12)
handles, labels = ax.get_legend_handles_labels()
ax.view_init(elev=20, azim=-45)

plt.title('3D PCA Trajectories with Temporal Dynamics', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
ncols = 2
nrows = len(groups) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 8))
axes = axes.flatten()
for group_idx in range(len(groups)):
    for pc_idx in range(n_comps):
        axes[group_idx].plot(time_pts, pc_timeseries[group_idx, pc_idx, :], color=color_list[pc_idx], label='PC%d' % (pc_idx+1))
        axes[group_idx].axhline(0, color='k', linestyle='--', alpha=0.5)
        axes[group_idx].axvline(0, color='k', linestyle='--', alpha=0.5)
        axes[group_idx].axvline(0.2, color='b', linestyle='--', alpha=0.5)
    # rThis = (pc_timeseries[group_idx, 0, :]**2 + pc_timeseries[group_idx, 1, :]**2)**0.5
    # axes[group_idx].plot(time_pts, rThis, color='k', label='PCradius')
    # axes[group_idx].axhline(1, color='k', linestyle='--', alpha=0.5)
    # axes[group_idx].axvline(0, color='k', linestyle='--', alpha=0.5)



    axes[group_idx].set_title(list(groups.keys())[group_idx])
    axes[group_idx].set_xlabel('Time (s)')
    axes[group_idx].set_ylabel('PC Amplitude')
    axes[group_idx].legend()

In [ ]:
plt.figure()
plt.plot(freqs[0], aa)
plt.show()

In [ ]:
powspctrm_combined.shape

In [ ]:
times

In [ ]:
filtered_trial_arr, filtered_time_arr = filter_and_downsample(trial_arr, time_arr, Fs)
pc_timeseries, groups = generate_pc_timeseries(filtered_trial_arr, filtered_time_arr, trialinfo[:,1], n_groups=2, n_components=2)

# a = np.nanmean(filtered_trial_arr, axis=0)
# base_a = np.nanmean(filtered_trial_arr[:, :, np.where(filtered_time_arr < 0)[0]], axis=(0, 2))
# base_a = np.repeat(base_a[:, np.newaxis], filtered_trial_arr.shape[2], axis=1)
# a = a - base_a
# # Plot in 3D
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# X, Y = np.meshgrid(filtered_time_arr, range(a.shape[0]))
# ax.plot_surface(X, Y, a, cmap='viridis')
# plt.show()

In [ ]:
# Create a 3D plot of the first 2 PCs
fig = plt.figure(figsize = (12, 8))
ax = fig.add_subplot(111, projection='3d')
color_list = ['r', 'g', 'b', 'y']
colors = [color_list[i] for i in range(len(groups))]
time_pts = filtered_time_arr

for group_idx in range(len(groups)):
    x = pc_timeseries[group_idx, 0, :]
    y = pc_timeseries[group_idx, 1, :]
    z = time_pts

    sc = ax.scatter3D(
        x, y, z,
        c=colors[group_idx],
        marker='o',
        s=40,
        edgecolors=colors[group_idx],
        label=list(groups.keys())[group_idx]
    )

ax.plot([0, 0], [0, 0], [z.min(), z.max()], 'k--', alpha=0.5)
ax.plot([0, 0], [0, 0], [0, 0], 'ko', ms=8)

ax.set_xlabel('PC1', fontsize=12, labelpad=15)
ax.set_ylabel('PC2', fontsize=12, labelpad=15)
ax.set_zlabel('Time (s)', fontsize=12, labelpad=15)
ax.xaxis.pane.fill = True
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.grid(True, alpha=0.3)

ax.legend(loc='upper left', fontsize=12)
handles, labels = ax.get_legend_handles_labels()
ax.view_init(elev=20, azim=-45)

plt.title('3D PCA Trajectories with Temporal Dynamics', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(10, 5))
for grp_idx in range(4):
    ax[0].plot(filtered_time_arr, pc_timeseries[grp_idx, 0], label=list(groups.keys())[grp_idx])
    ax[1].plot(filtered_time_arr, pc_timeseries[grp_idx, 1], label=list(groups.keys())[grp_idx])
ax[0].set_title('PC1')
ax[1].set_title('PC2')
for a in ax:
    a.axvline(0, color='k', linestyle='--')
    a.axhline(0, color='k', linestyle='--')
    a.set_xlabel('Time (s)')
    a.legend()
plt.show()

In [ ]:
alpha_idx = np.where((freqs >= 15) & (freqs <= 25))[1]
time_idx = np.where((times >= 0) & (times <= 2))[1]
times_crop = times[0, time_idx]
powspctrm_left_alpha = np.squeeze(np.mean(powspctrm_left[:, :, alpha_idx, :], axis=2))
powspctrm_right_alpha = np.squeeze(np.mean(powspctrm_right[:, :, alpha_idx, :], axis=2))
powspctrm_left_alpha = powspctrm_left_alpha[:, :, time_idx]
powspctrm_right_alpha = powspctrm_right_alpha[:, :, time_idx]

In [ ]:
a.shape

In [ ]:
base_a.shape

## Time-frequency RSA
#### 1. Create a category-based RSA models: Model1: Each location is unique; Model 2: Each quadrant is unique; Model 3: Each hemisphere is unique
#### 2. Sort trials based on the model being tested
#### 3. For each time and frequency, compute the neural RSM 
#### 4. Compute similarity between neural RSM and category-based RSA model. How to test similary? Maybe cosine similarity?
#### 5. Visualize time-frequency RSA for each model

In [ ]:
powspctrm_sorted.shape

In [ ]:
powspctrm_combined.shape

In [ ]:
from itertools import product
# Combine the data from left and right groups
trialInfo_combined = np.concatenate((trialInfo_left, trialInfo_right), axis=0)
powspctrm_combined = np.concatenate((powspctrm_left, powspctrm_right), axis=0)

trlAvgPow = np.nanmean(powspctrm_combined, axis=0)
trlAvgPow = np.repeat(trlAvgPow[np.newaxis, :, :, :], powspctrm_combined.shape[0], axis=0)

# Subtract the average power across trials
powspctrm_combined = 10**(powspctrm_combined / 10) / 10**(trlAvgPow / 10)

# baselineTime = (-1, 0)
# baselineIdx = np.where((times >= baselineTime[0]) & (times < baselineTime[1]))[0]
# baseline = np.mean(powspctrm_combined[:, :, :, baselineIdx], axis=-1)
# baseline = np.repeat(baseline[:, :, :, np.newaxis], powspctrm_combined.shape[-1], axis=-1)
# powspctrm_combined = 10**(powspctrm_combined / 10) / 10**(baseline / 10)

### Create a categorical Model, choose between 1, 2, 3
leftIdx = [ch_label.index(ch) for ch in leftChans]
rightIdx = [ch_label.index(ch) for ch in rightChans]
chIdx = leftIdx + rightIdx
ModelIdx = 3
if ModelIdx == 1: # Each location is a category
    validVals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    validIdx = np.where(np.isin(trialInfo_combined[:, 1], validVals))[0]
    trlInfo_ = trialInfo_combined[validIdx, 1]
    trlIdx = np.argsort(trlInfo_)
    trlInfo_sorted = trlInfo_[trlIdx]
    
elif ModelIdx == 2: # Grouping by quadrant
    validVals = [2, 3, 4, 5, 7, 8, 9, 10]
    validIdx = np.where(np.isin(trialInfo_combined[:, 1], validVals))[0]
    trlInfo_ = trialInfo_combined[validIdx, 1]
    trlInfo_cat = np.select(
        condlist = [
            (trlInfo_ == 2) | (trlInfo_ == 3),
            (trlInfo_ == 4) | (trlInfo_ == 5),
            (trlInfo_ == 7) | (trlInfo_ == 8),
            (trlInfo_ == 9) | (trlInfo_ == 10)
        ],
        choicelist = [1, 2, 3, 4],
        default = 0
    )
    trlIdx = np.argsort(trlInfo_cat)
    trlInfo_sorted = trlInfo_cat[trlIdx]

elif ModelIdx == 3: # Grouping by hemisphere
    validVals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    validIdx = np.where(np.isin(trialInfo_combined[:, 1], validVals))[0]
    trlInfo_ = trialInfo_combined[validIdx, 1]
    trlInfo_cat = np.select(
        condlist = [
            (trlInfo_ == 1) | (trlInfo_ == 2) | (trlInfo_ == 3) | (trlInfo_ == 9) | (trlInfo_ == 10),
            (trlInfo_ == 4) | (trlInfo_ == 5) | (trlInfo_ == 6) | (trlInfo_ == 7) | (trlInfo_ == 8)
        ],
        choicelist = [1, 2],
        default = 0
    )
    trlIdx = np.argsort(trlInfo_cat)
    trlInfo_sorted = trlInfo_cat[trlIdx]
    
powspctrm_ = powspctrm_combined[validIdx, :, :, :]
powspctrm_sorted = powspctrm_[trlIdx, :, :, :]
powspctrm_sorted = powspctrm_sorted[:, chIdx, :, :]

CatModel = np.zeros((len(trlInfo_sorted), len(trlInfo_sorted)))
for i, j in product(range(len(trlInfo_sorted)), range(len(trlInfo_sorted))):
    if trlInfo_sorted[i] == trlInfo_sorted[j]:
        CatModel[i, j] = 1


NeuralRSM = np.zeros((len(trlInfo_sorted), len(trlInfo_sorted), powspctrm_sorted.shape[2], powspctrm_sorted.shape[3]))
for f, t in product(range(powspctrm_sorted.shape[2]), range(powspctrm_sorted.shape[3])):
    NeuralRSM[:, :, f, t] = np.corrcoef(powspctrm_sorted[:, :, f, t])

In [ ]:
trlInfo_sorted

In [ ]:
np.where(trlInfo_sorted == 1)[0][0], np.where(trlInfo_sorted == 2)[0][0]#, np.where(trlInfo_sorted == 3)[0][0], np.where(trlInfo_sorted == 4)[0][0]

In [ ]:
# Print value counts for trlInfo_combined
np.unique(trialInfo_combined[:, 1], return_counts=True)


In [ ]:
trialInfo_combined = np.concatenate((trialInfo_left, trialInfo_right), axis=0)
powspctrm_combined = np.concatenate((powspctrm_left, powspctrm_right), axis=0)
tt = trialInfo_combined[:, 1]

powMat1 = powspctrm_combined[np.where((tt == 1) |
                                      (tt == 2) |
                                      (tt == 3) |
                                      (tt == 9) |
                                      (tt == 10))[0], :, :, :]
powMat2 = powspctrm_combined[np.where((tt == 4) |
                                        (tt == 5) |
                                        (tt == 6) |
                                        (tt == 7) |
                                        (tt == 8))[0], :, :, :]
avgMat = np.nanmean(powspctrm_combined, axis=0)

powMat1_norm = 10**(powMat1 / 10) / 10**(avgMat / 10)
powMat2_norm = 10**(powMat2 / 10) / 10**(avgMat / 10)

In [ ]:
# Select channels, time and freqs
freqIdx = np.where((freqs >= 5) & (freqs <= 50))[0]
timeIdx = np.where((times >= -1) & (times <= 2))[0]
p1 = powMat2_norm[:, leftIdx, :, :]
p1 = p1[:, :, freqIdx, :]
p1 = p1[:, :, :, timeIdx]
p1 = np.nanmean(p1, axis=(0,1))

p2 = powMat2_norm[:, rightIdx, :, :]
p2 = p2[:, :, freqIdx, :]
p2 = p2[:, :, :, timeIdx]
p2 = np.nanmean(p2, axis=(0,1))

pp = p2 - p1
f, axs = plt.subplots(2, 3, figsize=(10, 5))
axs = axs.flatten()
axs[0].hist(pp.flatten(), bins=100)
axs[1].imshow(pp, aspect='auto')
axs[2].imshow(p1, aspect='auto')
axs[3].imshow(p2, aspect='auto')
# plt.gca().invert_yaxis()

axs[4].plot(freqs[freqIdx], np.nanmean(pp, axis=1))
axs[5].plot(freqs[freqIdx], np.nanmean(p1, axis=1), label='Left')
axs[5].plot(freqs[freqIdx], np.nanmean(p2, axis=1), label='Right')
axs[5].legend()

plt.show()


In [ ]:
times[50], times[70]

In [ ]:
powspctrm_sorted.shape

In [ ]:
tIdxArr = np.where((times >= 0.5) & (times <= 1))[0]
fIdxArr = np.where((freqs >= 8) & (freqs <= 12))[0]
pppp = np.concatenate((powMat1_norm, powMat2_norm), axis=0)
a_holder = np.empty((len(trlInfo_sorted), len(trlInfo_sorted), len(fIdxArr), len(tIdxArr)))
for f, t in product(range(len(fIdxArr)), range(len(tIdxArr))):
    tMat = pppp[:, :, fIdxArr[f], tIdxArr[t]]

    a_holder[:, :, f, t] = np.corrcoef(tMat)

a = np.nanmean(a_holder, axis=(2, 3))
qThresh = 20
lowBound = np.nanpercentile(a, qThresh)
upBound = np.nanpercentile(a, 100 - qThresh)
print(lowBound, upBound)
# a = np.tril(a, k=1)
f, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].hist(a.flatten(), bins=100)
axs[1].imshow(a, vmin=lowBound, vmax=upBound, cmap='RdBu_r')
# axs[1].plot(freqs, np.mean(a, axis=1), 'k')
plt.show()

In [ ]:
a.shape

In [ ]:
freqs[15]

In [ ]:
freqIdx = np.where((freqs >= 20) & (freqs <= 25))[0]
tSteps = np.arange(0, 2, 0.3)
ncols = 3
nrows = len(tSteps) // ncols + 1
f, axs = plt.subplots(nrows, ncols, figsize=(nrows*5, ncols*5))
axs = axs.flatten()
for i, t in enumerate(tSteps):
    timeIdx = np.where((times >= t) & (times < t+0.2))[0]
    a1 = NeuralRSM[:, :, :, timeIdx]
    a2 = NeuralRSM[:, :, freqIdx, :]
    a = np.nanmean(a2, axis=(2, 3))

    axs[i].imshow(a, cmap='RdBu_r', vmin=np.quantile(np.tril(a, -1).flatten(), 0.51), vmax=np.quantile(np.tril(a, -1).flatten(), 0.9))
    axs[i].set_title('Time: %.1f - %.1f' % (t, t+0.2))
plt.show()

# print(a.shape)
# plt.figure()
# plt.imshow(a, cmap='RdBu_r', vmin=np.quantile(a, 0.2), vmax=np.quantile(a, 0.8))
# plt.colorbar()
# plt.show()

## Decode Target Location based on saccade error


In [ ]:
bidsRoot = '/System/Volumes/Data/d/DATD/datd/MEG_MGS/MEG_BIDS'
taskName = 'mgs'

subjID = 1
subName = 'sub-%02d' % subjID
derivativesRoot = os.path.join(bidsRoot, 'derivatives', subName)
eyeRoot = os.path.join(derivativesRoot, 'eyetracking')
megRoot = os.path.join(derivativesRoot, 'meg')
stimRoot = os.path.join(bidsRoot, subName, 'stimfiles')
fNameRoot = subName + '_task-' + taskName
iisess_fpath = os.path.join(eyeRoot, fNameRoot + '-iisess.mat')
stimLocked_fpath = os.path.join(megRoot, fNameRoot + '_stimlocked.mat')
TFR_fpath = os.path.join(megRoot, fNameRoot + '_TFR.mat')


# Load ii_sess data and TFR data
ii_sess = loadmat(iisess_fpath)['ii_sess']
epocStimLocked = loadmat(stimLocked_fpath)
TFRmat = loadmat(TFR_fpath)

# Extract epocData
ch_label = epocStimLocked['epocStimLocked'][0][0][0]
trialinfo = epocStimLocked['epocStimLocked'][0][0][1]
sampleinfo = epocStimLocked['epocStimLocked'][0][0][2]
trial = epocStimLocked['epocStimLocked'][0][0][4][0]
trials_list = [np.array(t) for t in trial]
trial_arr = np.stack(trials_list, axis=0)
time = epocStimLocked['epocStimLocked'][0][0][5]
time_arr = time[0][0][0]
Fs = epocStimLocked['epocStimLocked'][0][0][6][0][0]

# Extract TFR data
# ch_label_array = TFRmat['TFRleft_power'][0][0][0]
# ch_label = []
# for i in range(len(ch_label_array)):
#     ch_label.append(ch_label_array[i][0][0])
# dimord = TFRmat['TFRleft_power'][0][0][1]
# freqs = TFRmat['TFRleft_power'][0][0][2][0]
# times = TFRmat['TFRleft_power'][0][0][3][0]
trialInfo_left = TFRmat['TFRleft_power'][0][0][5]
powspctrm_left = TFRmat['TFRleft_power'][0][0][7]
trialInfo_right = TFRmat['TFRright_power'][0][0][5]
powspctrm_right = TFRmat['TFRright_power'][0][0][7]

In [ ]:
## Visualize errors across all subjects
import seaborn as sns
subList = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 
               22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
# subList = [1, 2]

mean_ierr = np.zeros((len(subList), 2))
mean_ferr = np.zeros((len(subList), 2))
mean_iRT = np.zeros((len(subList), 2))
mean_fRT = np.zeros((len(subList), 2))

for sIdx, subjID in enumerate(subList):
    subName = 'sub-%02d' % subjID
    derivativesRoot = os.path.join(bidsRoot, 'derivatives', subName)
    eyeRoot = os.path.join(derivativesRoot, 'eyetracking')
    megRoot = os.path.join(derivativesRoot, 'meg')
    stimRoot = os.path.join(bidsRoot, subName, 'stimfiles')
    fNameRoot = subName + '_task-' + taskName
    iisess_fpath = os.path.join(eyeRoot, fNameRoot + '-iisess.mat')
    ii_sess = loadmat(iisess_fpath)['ii_sess']

    # Extract relevant data
    ierr = ii_sess['i_sacc_err'][0, 0].T[0]
    ferr = ii_sess['f_sacc_err'][0, 0].T[0]
    irt = ii_sess['i_sacc_rt'][0, 0].T[0]
    frt = ii_sess['f_sacc_rt'][0, 0].T[0]
    tarlocGaze = ii_sess['tarlocCode'][0, 0].T[0]

    left_trials = []
    right_trials = []
    for ijk in range(trial_arr.shape[0]):
        if trialinfo[ijk, 1] == 4 or trialinfo[ijk, 1] == 5 or trialinfo[ijk, 1] == 6 or trialinfo[ijk, 1] == 7 or trialinfo[ijk, 1] == 8:
            # Check if there is any nan in the data
            if ~np.isnan(trial_arr[ijk, :, :]).any():
                left_trials.append(ijk+1)
        elif trialinfo[ijk, 1] == 1 or trialinfo[ijk, 1] == 2 or trialinfo[ijk, 1] == 3 or trialinfo[ijk, 1] == 9 or trialinfo[ijk, 1] == 10:

            if ~np.isnan(trial_arr[ijk, :, :]).any():
                right_trials.append(ijk+1)

    left_ierr = ierr[left_trials]
    right_ierr = ierr[right_trials]
    left_ferr = ferr[left_trials]
    right_ferr = ferr[right_trials]
    left_irt = irt[left_trials]
    right_irt = irt[right_trials]
    left_frt = frt[left_trials]
    right_frt = frt[right_trials]
    

    mean_ierr[sIdx-1, 0] = np.nanmean(left_ierr)
    mean_ierr[sIdx-1, 1] = np.nanmean(right_ierr)
    mean_ferr[sIdx-1, 0] = np.nanmean(left_ferr)
    mean_ferr[sIdx-1, 1] = np.nanmean(right_ferr)
    mean_iRT[sIdx-1, 0] = np.nanmean(left_irt)
    mean_iRT[sIdx-1, 1] = np.nanmean(right_irt)
    mean_fRT[sIdx-1, 0] = np.nanmean(left_frt)
    mean_fRT[sIdx-1, 1] = np.nanmean(right_frt)

    del ii_sess, ierr, ferr, irt, frt, tarlocGaze, left_trials, right_trials

mean_mean_ierr = np.nanmean(mean_ierr, axis=0)
sem_mean_ierr = np.nanstd(mean_ierr, axis=0) / np.sqrt(mean_ierr.shape[0])
mean_mean_ferr = np.nanmean(mean_ferr, axis=0)
sem_mean_ferr = np.nanstd(mean_ferr, axis=0) / np.sqrt(mean_ferr.shape[0])
mean_mean_iRT = np.nanmean(mean_iRT, axis=0)
sem_mean_iRT = np.nanstd(mean_iRT, axis=0) / np.sqrt(mean_iRT.shape[0])
mean_mean_fRT = np.nanmean(mean_fRT, axis=0)
sem_mean_fRT = np.nanstd(mean_fRT, axis=0) / np.sqrt(mean_fRT.shape[0])

f, axs = plt.subplots(2, 2, figsize=(10, 10))
axs = axs.flatten()
sns.boxplot(data=mean_ierr, ax=axs[0])
axs[0].set_xticklabels(['Left', 'Right'])
axs[0].set_title('Initial Error')
sns.boxplot(data=mean_ferr, ax=axs[1])
axs[1].set_xticklabels(['Left', 'Right'])
axs[1].set_title('Final Error')
sns.boxplot(data=mean_iRT, ax=axs[2])
axs[2].set_xticklabels(['Left', 'Right'])
axs[2].set_title('Initial RT')
sns.boxplot(data=mean_fRT, ax=axs[3])
axs[3].set_xticklabels(['Left', 'Right'])
axs[3].set_title('Final RT')
plt.show()

In [ ]:
left_trials = []
right_trials = []
for ijk in range(trial_arr.shape[0]):
    if trialinfo[ijk, 1] == 4 or trialinfo[ijk, 1] == 5 or trialinfo[ijk, 1] == 6 or trialinfo[ijk, 1] == 7 or trialinfo[ijk, 1] == 8:
        # Check if there is any nan in the data
        if ~np.isnan(trial_arr[ijk, :, :]).any():
            left_trials.append(ijk+1)
    elif trialinfo[ijk, 1] == 1 or trialinfo[ijk, 1] == 2 or trialinfo[ijk, 1] == 3 or trialinfo[ijk, 1] == 9 or trialinfo[ijk, 1] == 10:

        if ~np.isnan(trial_arr[ijk, :, :]).any():
            right_trials.append(ijk+1)
            

In [ ]:
ierr = ii_sess['i_sacc_err'][0, 0].T[0]
ferr = ii_sess['f_sacc_err'][0, 0].T[0]
irt = ii_sess['i_sacc_rt'][0, 0].T[0]
frt = ii_sess['f_sacc_rt'][0, 0].T[0]
tarlocGaze = ii_sess['tarlocCode'][0, 0].T[0]

In [ ]:
left_err = ierr[left_trials]
right_err = ierr[right_trials]
all_err = np.concatenate((left_err, right_err))
plt.figure()
plt.hist(all_err, bins=1000)
plt.show()

In [ ]:
len(np.where(all_err >= 1e-1)[0])

In [ ]:
len(tarlocGaze), powspctrm_left.shape, powspctrm_right.shape